processed_v2 link:
  https://docs.google.com/spreadsheets/d/1IgQmuxmhJ6uDlRuGMvf2pD_U1W9FJfV2/edit?usp=sharing&ouid=105650582918007922218&rtpof=true&sd=true

labels link:
  https://docs.google.com/spreadsheets/d/1uHqfgMvsLMgozd074EGP5XAbkG8EzahT/edit?usp=sharing&ouid=105650582918007922218&rtpof=true&sd=true

requirements: pandas, sclearn, numpy

In [16]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
df_text = pd.read_csv('drive/MyDrive/Colab Notebooks/NLP/processed_v2.xls')
df_labels = pd.read_csv('drive/MyDrive/Colab Notebooks/NLP/labels.xls')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
from sklearn.model_selection import train_test_split

# combine features and target
df = df_text.copy()
df['label'] = df_labels.iloc[:, 0]

# first split: train (70%) and temp (30%)
df_train, df_temp = train_test_split(
    df,
    test_size=0.3,
    random_state=42,
    stratify=df['label']
)

# second split: validation (15%) and test (15%)
df_val, df_test = train_test_split(
    df_temp,
    test_size=0.5,
    random_state=42,
    stratify=df_temp['label']
)

# separate X and y
X_train = df_train[['statement_1', 'statement_2']]
y_train = df_train['label']

X_val = df_val[['statement_1', 'statement_2']]
y_val = df_val['label']

X_test = df_test[['statement_1', 'statement_2']]
y_test = df_test['label']

# check shapes
print(X_train.shape, X_val.shape, X_test.shape)

(1747, 2) (375, 2) (375, 2)


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def make_pairs(X):
    return X['statement_1'].values, X['statement_2'].values

s1_train, s2_train = make_pairs(X_train)
s1_val,   s2_val   = make_pairs(X_val)
s1_test,  s2_test  = make_pairs(X_test)

# fit vectorizer on TRAIN ONLY
vectorizer = TfidfVectorizer()
vectorizer.fit(list(s1_train) + list(s2_train))

def predict_threshold(s1, s2, vectorizer, threshold):
    v1 = vectorizer.transform(s1)
    v2 = vectorizer.transform(s2)
    sim = cosine_similarity(v1, v2).diagonal()
    return (sim >= threshold).astype(int)

# ---- tune threshold on validation ----
thresholds = np.linspace(0.1, 0.9, 17)

best_score = -1
best_thr = None

for thr in thresholds:
    y_val_pred = predict_threshold(s1_val, s2_val, vectorizer, thr)

    acc = accuracy_score(y_val, y_val_pred)
    f1  = f1_score(y_val, y_val_pred, average='macro')
    score = (acc + f1) / 2

    if score > best_score:
        best_score = score
        best_thr = thr

print("Best threshold:", best_thr)

# ---- final evaluation on test ----
y_test_pred = predict_threshold(s1_test, s2_test, vectorizer, best_thr)

print("TF-IDF BASELINE")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("Macro-F1:", f1_score(y_test, y_test_pred, average='macro'))

Best threshold: 0.2
TF-IDF BASELINE
Accuracy: 0.6826666666666666
Macro-F1: 0.627780697144859


In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def extract_features(X):
    s1 = X['statement_1'].values
    s2 = X['statement_2'].values

    feats = []
    for a, b in zip(s1, s2):
        set_a = set(a.split())
        set_b = set(b.split())

        len_a = len(a)
        len_b = len(b)
        overlap = len(set_a & set_b)
        union = len(set_a | set_b) if len(set_a | set_b) > 0 else 1

        jaccard = overlap / union

        feats.append([len_a, len_b, overlap, jaccard])

    return np.array(feats)

X_train_f = extract_features(X_train)
X_val_f   = extract_features(X_val)
X_test_f  = extract_features(X_test)

# ---- tune LR ----
param_grid = {
    'C': [0.1, 1.0, 10.0]
}

best_score = -1
best_params = None

for params in ParameterGrid(param_grid):
    model = LogisticRegression(max_iter=1000, **params)
    model.fit(X_train_f, y_train)

    y_val_pred = model.predict(X_val_f)

    acc = accuracy_score(y_val, y_val_pred)
    f1  = f1_score(y_val, y_val_pred, average='macro')
    score = (acc + f1) / 2

    if score > best_score:
        best_score = score
        best_params = params

print("Best LR params:", best_params)

# ---- retrain on train+val ----
X_train_val_f = np.vstack([X_train_f, X_val_f])
y_train_val = np.concatenate([y_train, y_val])

final_model = LogisticRegression(max_iter=1000, **best_params)
final_model.fit(X_train_val_f, y_train_val)

# ---- test ----
y_test_pred = final_model.predict(X_test_f)

print("\nLOGISTIC REGRESSION BASELINE")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("Macro-F1:", f1_score(y_test, y_test_pred, average='macro'))

Best LR params: {'C': 10.0}

LOGISTIC REGRESSION BASELINE
Accuracy: 0.672
Macro-F1: 0.5521366359514123


In [20]:
from sklearn.metrics import confusion_matrix

cm_tfidf = confusion_matrix(y_test, y_test_pred)
print("TF-IDF Confusion Matrix:")
print(cm_tfidf)

TF-IDF Confusion Matrix:
[[223  27]
 [ 96  29]]


In [21]:
cm_lr = confusion_matrix(y_test, y_test_pred)
print("\nLogistic Regression Confusion Matrix:")
print(cm_lr)


Logistic Regression Confusion Matrix:
[[223  27]
 [ 96  29]]


In [22]:
import pandas as pd

def print_cm(cm, name):
    df_cm = pd.DataFrame(cm,
                         index=['actual_0','actual_1'],
                         columns=['pred_0','pred_1'])
    print(f"\n{name}")
    print(df_cm)

print_cm(cm_tfidf, "TF-IDF Confusion Matrix")
print_cm(cm_lr, "LogReg Confusion Matrix")


TF-IDF Confusion Matrix
          pred_0  pred_1
actual_0     223      27
actual_1      96      29

LogReg Confusion Matrix
          pred_0  pred_1
actual_0     223      27
actual_1      96      29


In [23]:
import numpy as np

feature_names = ['len_1', 'len_2', 'overlap', 'jaccard']

coefs = final_model.coef_[0]

feat_imp = pd.DataFrame({
    'feature': feature_names,
    'coef': coefs,
    'abs_coef': np.abs(coefs)
}).sort_values('abs_coef', ascending=False)

print("Top features:")
print(feat_imp.head(10))

Top features:
   feature      coef  abs_coef
3  jaccard  2.805198  2.805198
2  overlap -0.023749  0.023749
0    len_1 -0.008820  0.008820
1    len_2 -0.000246  0.000246


In [34]:
import pandas as pd

# create dataframe with predictions
df_errors = X_test.copy()
df_errors = df_errors.reset_index(drop=True)

df_errors['true'] = y_test.values
df_errors['pred'] = y_test_pred

# filter errors
df_errors = df_errors[df_errors['true'] != df_errors['pred']]

# show 10 examples
errors_sample = df_errors[16:26]
print(errors_sample)

                           statement_1                       statement_2  \
75                       Зима пройшла.                      Зима минула.   
79                США - велика країна.  Сполучені Штати - велика країна.   
80       Він зробив фотографію родини.    Легко знайшов - легко втратив.   
88             Ця краватка тобі пасує.         Та краватка тобі до лиця.   
89                 Тому потрібна Мері.               Мері потрібна Тому.   
92        Хто наглядає за цим собакою?        Хто доглядає цього собаку?   
93                У Тома нічого немає.                Том нічого не має.   
96   Три в кубі дорівнює двадцять сім.             3 в кубі дорівнює 27.   
99                          Я не знаю.                   Мені не відомо.   
100             Я вивчаю перську мову.                   Я вивчаю фарсі.   

     true  pred  
75      1     0  
79      0     1  
80      1     0  
88      1     0  
89      1     0  
92      1     0  
93      1     0  
96      1     0  
9

In [35]:
# save misclassified examples to JSONL
errors_sample.reset_index(drop=True).to_json(
    "error_cases_lab6.jsonl",
    orient="records",
    lines=True
)
print("Saved 10 error examples to 'error_cases_lab6.jsonl'")

Saved 10 error examples to 'error_cases_lab6.jsonl'
